In [0]:
%run ./01-config

In [0]:
class Bronze():
    def __init__(self):        
        self.Conf = Config()
        self.landing_zone = self.Conf.base_dir_data + "/raw" 
        self.checkpoint_base = self.Conf.base_dir_checkpoint + "/checkpoints"
        self.catalog = self.Conf.catalog
        self.db_name = self.Conf.db_name
        spark.sql(f"USE {self.catalog}.{self.db_name}")
        
    def consume_user_registration(self, once=True, processing_time=None):
        from pyspark.sql import functions as F
        schema = "user_id long, device_id long, mac_address string, registration_timestamp double"
        
        df_stream = (
                        spark.readStream
                        .format("csv")
                        .schema(schema)
                        .option("header", "true")
                        .option("maxFilesPerTrigger", 1)
                        .load(f"{self.landing_zone}/registered_users_bz")
                        .withColumn("load_time",F.current_timestamp())
                        .withColumn("source_file",F.col("_metadata.file_path"))
                    )
                        
        query = (
                    df_stream.writeStream
                    .format("delta")
                    .option("checkpointLocation",f"{self.checkpoint_base}/registered_users_bz")
                    .outputMode("append")
                    .queryName("registered_users_bz_ingestion")
                    .trigger(availableNow=True)
                    .toTable(f"{self.catalog}.{self.db_name}.registered_users_bz")
                )

        return query
          
    def consume_gym_logins(self, once=True, processing_time=None):
        from pyspark.sql import functions as F
        schema = "mac_address string, gym bigint, login double, logout double"
        
        df_stream = (
                        spark.readStream
                        .format("csv")
                        .schema(schema)
                        .option("header", "true")
                        .option("maxFilesPerTrigger", 1)
                        .load(f"{self.landing_zone}/gym_logins_bz")
                        .withColumn("load_time",F.current_timestamp())
                        .withColumn("source_file",F.col("_metadata.file_path"))
                    )

        query = (
                    df_stream.writeStream
                    .format("delta")
                    .option("checkpointLocation",f"{self.checkpoint_base}/gym_logins_bz")
                    .outputMode("append")
                    .queryName("gym_logins_bz_ingestion")
                    .trigger(availableNow=True)
                    .toTable(f"{self.catalog}.{self.db_name}.gym_logins_bz")
                )

        return query
        
        
    def consume_kafka_multiplex(self, once=True, processing_time=None):
        from pyspark.sql import functions as F

        Conf = Config()
        date_lookup = (
                            spark.table(f"{self.catalog}.{self.db_name}.date_lookup")
                                .select("date", "week_part")
                        )

        kafka_df = (
                        spark.readStream
                            .format("kafka")
                            .option("kafka.bootstrap.servers",Conf.BOOTSTRAP_SERVER)
                            .option("kafka.security.protocol", "SASL_SSL").option("kafka.sasl.mechanism", "PLAIN")
                            .option("kafka.sasl.jaas.config",f"{Conf.JAAS_MODULE} required "
                                    f"username='{Conf.CLUSTER_API_KEY}' "
                                    f"password='{Conf.CLUSTER_API_SECRET}';")
                            .option("subscribe","user_info,workout,bpm")
                            .option("startingOffsets", "earliest")
                            .option("maxOffsetsPerTrigger", 10000)
                            .load()
                    )

        df_stream = (
                        kafka_df.select(
                                F.col("key").cast("string").alias("key"),
                                F.col("value").cast("string").alias("value"),
                                "topic",
                                "partition",
                                "offset",
                                (F.col("timestamp").cast("long") * 1000).cast("long").alias("timestamp"))
                                .withColumn("event_timestamp",F.to_timestamp(F.from_unixtime(F.col("timestamp") / 1000)))
                                .withColumn("date",F.to_date("event_timestamp"))
                                .join(F.broadcast(date_lookup),["date"],"left")
                                .drop("event_timestamp")
                                .withColumn("load_time",F.current_timestamp())
                                .withColumn("source_file",F.lit(None).cast("string"))
                    )

        query = (
                    df_stream.writeStream
                            .format("delta")
                            .option("checkpointLocation",f"{self.checkpoint_base}/kafka_multiplex_bz")
                            .outputMode("append")
                            .queryName("kafka_multiplex_bz_ingestion")
                            .trigger(availableNow=True)
                            .toTable(f"{self.catalog}.{self.db_name}.kafka_multiplex_bz")
                )

        return query
        
    
    def consume(self):
        print("Starting Bronze ingestion...")
        queries = [
                    self.consume_user_registration(),
                    self.consume_gym_logins(),
                    self.consume_kafka_multiplex()
                ]
        for query in queries:
            query.awaitTermination()
        print("Bronze ingestion completed.")
        
        
    def assert_count(self, table_name, expected_count, filter="true"):
        print(f"Validating record counts in {table_name}...", end='')
        actual_count = spark.read.table(f"{self.catalog}.{self.db_name}.{table_name}").where(filter).count()
        assert actual_count == expected_count, f"Expected {expected_count:,} records, found {actual_count:,} in {table_name} where {filter}" 
        print(f"Found {actual_count:,} / Expected {expected_count:,} records where {filter}: Success")        
        
    def validate(self, sets):
        import time
        start = int(time.time())
        print(f"\nValidating bronz layer records...")
        self.assert_count("registered_users_bz", 5 if sets == 1 else 10)
        self.assert_count("gym_logins_bz", 8 if sets == 1 else 16)
        self.assert_count("kafka_multiplex_bz", 7 if sets == 1 else 13, "topic='user_info'")
        self.assert_count("kafka_multiplex_bz", 16 if sets == 1 else 32, "topic='workout'")
        self.assert_count("kafka_multiplex_bz", sets * 253801, "topic='bpm'")
        print(f"Bronze layer validation completed in {int(time.time()) - start} seconds")     